# 02️⃣ Feature Engineering

Compute derived indicators grouped by stock Symbol (Moving Averages, RSI, MACD, Bollinger Bands, Volatility, Momentum) on the NIFTY-50 dataset.

In [1]:
import os, pathlib, pandas as pd, numpy as np
%matplotlib inline


In [2]:
NOTEBOOK_DIR = pathlib.Path(os.path.abspath('') if '__file__' not in locals() else os.path.dirname(__file__))
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_PARQUET = PROJECT_ROOT / 'data' / 'parquet'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

files = list(DATA_PARQUET.glob('*.parquet'))
nifty_all = [f for f in files if 'NIFTY50_all' in f.name]
target_file = nifty_all[0] if nifty_all else files[0]

print(f'Loading data from {target_file.name}')
df = pd.read_parquet(target_file)
df['Date'] = pd.to_datetime(df['Date'])

# Crucial: Sort by Symbol and Date first!
df = df.sort_values(['Symbol', 'Date'])

# Daily returns per stock (grouped by Symbol)
df['Return'] = df.groupby('Symbol')['Close'].pct_change()

# 1. Moving Averages (grouped by Symbol)
df['MA20'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(window=20).mean())
df['MA50'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(window=50).mean())

# 2. Exponential Moving Averages (grouped by Symbol)
df['EMA20'] = df.groupby('Symbol')['Close'].transform(lambda x: x.ewm(span=20, adjust=False).mean())

# 3. Volatility (20-day rolling standard deviation of returns per stock)
df['Volatility'] = df.groupby('Symbol')['Return'].transform(lambda x: x.rolling(window=20).std())

# 4. Momentum (10-day rate of change per stock)
df['Momentum'] = df.groupby('Symbol')['Close'].transform(lambda x: x.diff(10))

# 5. Relative Strength Index (RSI) per stock
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))

df['RSI'] = df.groupby('Symbol')['Close'].transform(lambda x: compute_rsi(x))

# 6. MACD per stock
def compute_macd(series):
    ema12 = series.ewm(span=12, adjust=False).mean()
    ema26 = series.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    return pd.DataFrame({'MACD': macd, 'Signal_Line': signal}, index=series.index)

# Apply MACD per group and align
macd_df = df.groupby('Symbol')['Close'].apply(compute_macd).reset_index(level=0, drop=True)
df['MACD'] = macd_df['MACD']
df['Signal_Line'] = macd_df['Signal_Line']

# 7. Bollinger Bands per stock
df['BB_Middle'] = df['MA20']
df['BB_Std'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(window=20).std())
df['BB_Upper'] = df['BB_Middle'] + (2 * df['BB_Std'])
df['BB_Lower'] = df['BB_Middle'] - (2 * df['BB_Std'])

df.to_parquet(DATA_PARQUET / 'feature_engineered.parquet', index=False)
print('Feature engineered file saved. Columns shape:', df.shape)


Loading data from NIFTY50_all.parquet


Feature engineered file saved. Columns shape: (235192, 28)
